Configure the Spark session to include the required Cosmos DB Spark connector JARs (packages) via %%configure. This enables Spark to connect to Azure Cosmos DB.

In [ ]:
%%configure -f
{
    "conf": {
        "spark.jars.packages": "com.azure.cosmos.spark:azure-cosmos-spark_3-5_2-12:4.41.0,com.azure.cosmos.spark:fabric-cosmos-spark-auth_3:1.1.0"
    }
}

Copy the Endpoint for Cosmos DB NoSQL database under Settings > Connections and paste in the cell below, along with the database name and container name

In [ ]:
cosmos_endpoint = ""
cosmos_database = ""
cosmos_container = ""

Load the Delta table of animated_movies_with_embeddings from the Lakehouse into a Spark DataFrame named animated_movies_with_embeddings_df.

In [ ]:
animated_movies_with_embeddings_df = (
    spark.read
         .format("delta")
         .table("animated_movies_with_embeddings")   
)


Add a new id column to the DataFrame by generating a SHA-256 hash of the movieId. 

In [ ]:
from pyspark.sql.functions import col, sha2, concat_ws

animated_movies_with_embeddings_df = (
    animated_movies_with_embeddings_df
    .withColumn(
        "id",
        sha2(concat_ws("_", col("movieId")), 256)
    )    
)

Prepare a configuration dictionary (cosmos_config) with necessary connection parameters for writing to Cosmos DB via Spark. This includes endpoint, database/container names, authentication method (AAD token), and other required settings.

In [ ]:
cosmos_config = {
  "spark.cosmos.accountEndpoint": cosmos_endpoint,
  "spark.cosmos.database": cosmos_database,
  "spark.cosmos.container": cosmos_container,
  "spark.cosmos.accountDataResolverServiceName": "com.azure.cosmos.spark.fabric.FabricAccountDataResolver",
  "spark.cosmos.auth.type": "AccessToken",
  "spark.cosmos.useGatewayMode": "true",
  "spark.cosmos.auth.aad.audience": "https://cosmos.azure.com/",
}

Write the animated_movies_with_embeddings_df DataFrame to the specified Cosmos DB container, using the configuration prepared above, in append mode. This uploads each DataFrame row as a document in Cosmos DB.

In [ ]:
(
animated_movies_with_embeddings_df
    .write
    .format("cosmos.oltp")
    .options(**cosmos_config)
    .mode("APPEND")
    .save()
)